In [50]:
# Load libraries and data
import pandas as pd

# Load the merged data
data = pd.read_csv('../data/raw/item_level_data.csv')
data.head(3)

,participant_id,condition,imageid,headline,before_AI_Real_Fake,confidence_rating_before_AI,with_AI_Real_Fake,confidence_rating_with_AI,ground_truth_with_AI,after_AI_seen_before,after_AI_ground_truth,after_AI_Real_Fake,confidence_rating_after_AI,with_AI_seen_before,week
0,560379c9e372c00011bd4a90,Persuasive,0,"Senator Christopher J. Dodd, with his wife and...",FAKE,100,FAKE,100,fake,No,fake,Fake,100,No,week 0
1,560379c9e372c00011bd4a90,Persuasive,5,"Mikerlyne Dorvil, accused of causing the earth...",FAKE,100,FAKE,100,fake,No,fake,Fake,100,No,week 0
2,560379c9e372c00011bd4a90,Persuasive,7,A crowd gathered on Tuesday in Minneapolis aft...,REAL,48,REAL,100,real,No,real,Real,80,No,week 0


In [51]:
# Remove people who failed attention checks

# Load AI knowledge data
ai_knowledge = pd.read_csv('../data/raw/AI_knowledge.csv')
ai_knowledge = ai_knowledge.rename(columns={ai_knowledge.columns[0]: 'participant_id'})

# Define the AI literacy questions
ai_literacy_cols = [
    'I understand the basic concepts of Artifical Intelligence',
    'I believe I can contribute to AI projects', 
    'I can judge the pros and cons of AI',
    'I keep up with the latest AI trends',
    'I am comfortable talking about AI with others',
    'I can think of news ways to use existing AI tools'
]

# Convert Likert scale responses to numeric values
likert_mapping = {
    'Strongly Disagree': 1,
    'Disagree': 2, 
    'Somewhat Disagree': 3,
    'Neutral': 4,
    'Somewhat Agree': 5,
    'Agree': 6,
    'Strongly Agree': 7
}

# Apply mapping to AI literacy columns
for col in ai_literacy_cols:
    ai_knowledge[col] = ai_knowledge[col].map(likert_mapping)

# Calculate AI literacy score as mean of the 6 questions (excluding invalid responses)
ai_knowledge['ai_literacy'] = ai_knowledge[ai_literacy_cols].mean(axis=1, skipna=True)
ai_vars = ai_knowledge[['participant_id', 'ai_usage', 'ai_image_usage', 'ai_literacy']].copy()
ai_vars.head(3)

,participant_id,ai_usage,ai_image_usage,ai_literacy
0,5faefcc22be93b11829e84df,0-2 hours a day,Rarely,4.500000
1,67db3dd91c05b7278a0aad44,0-2 hours a day,Once a week,5.833333
2,5f0a686635f72917e94c7a63,Once a month,Rarely,NaN


In [52]:
# Process data for plotting

# Merge AI knowledge data with main data
data = data.merge(ai_vars, on='participant_id', how='left')
print(f"Data shape after merging AI variables: {data.shape}")

# Calculate accuracy for before, with, after AI
data['before_accuracy'] = (data['before_AI_Real_Fake'].str.lower() == data['ground_truth_with_AI'].str.lower()).astype(int) #* data['confidence_rating_before_AI'] / 100
data['with_accuracy'] = (data['with_AI_Real_Fake'].str.lower() == data['ground_truth_with_AI'].str.lower()).astype(int) #* data['confidence_rating_with_AI'] / 100
data['after_accuracy'] = (data['after_AI_Real_Fake'].str.lower() == data['after_AI_ground_truth'].str.lower()).astype(int) #* data['confidence_rating_after_AI'] / 100

# Numeric belief for before, with, after AI (Fake 0, Real 1)
data['before_belief'] = (data['before_AI_Real_Fake'].str.lower() == 'real').astype(int)
data['with_belief'] = (data['with_AI_Real_Fake'].str.lower() == 'real').astype(int)
data['after_belief'] = (data['after_AI_Real_Fake'].str.lower() == 'real').astype(int)

# Reshape data for plotting
processed_df = []
for idx, row in data.iterrows():
    for time_point, acc_col, belief_col, conf_col, gt_col, seen_col in [('before', 'before_accuracy', 'before_belief', 'confidence_rating_before_AI', 'ground_truth_with_AI', 'with_AI_seen_before'), ('with', 'with_accuracy', 'with_belief', 'confidence_rating_with_AI', 'ground_truth_with_AI', 'with_AI_seen_before'), ('after', 'after_accuracy', 'after_belief', 'confidence_rating_after_AI', 'after_AI_ground_truth', 'after_AI_seen_before')]:
        processed_df.append({
            # for week just the int e.g. week 0 becomes 0
            'participant_id': row['participant_id'], 'condition': row['condition'], 'week': int(row['week'].split(' ')[1]), 'imageid': row['imageid'],
            'time_point': time_point, 'accuracy': row[acc_col], 'belief': row[belief_col], 'confidence': row[conf_col], 'ground_truth': row[gt_col].lower(), 'seen': row[seen_col]=='Yes',
            # Add AI knowledge variables
            'ai_usage': row['ai_usage'], 'ai_image_usage': row['ai_image_usage'], 'ai_literacy': row['ai_literacy']
        })
processed_df = pd.DataFrame(processed_df)

# Create new imageids to ensure uniqueness across weeks and time points
processed_df['imageid'] = processed_df.groupby(['week', 'time_point'])['imageid'].transform(lambda x: pd.factorize(x)[0]) + processed_df['week'] * 20 + (processed_df['time_point'] == 'after') * 100

# Remove people who are not in week 4
ids_w4 = set(processed_df[processed_df['week'] == 4]['participant_id'].unique())
processed_df = processed_df[processed_df['participant_id'].isin(ids_w4)]

# Remove people who failed attention checks
ids_failed_attention = pd.read_csv('../data/raw/attention_check_failed.csv')['participant_id'].tolist()
processed_df = processed_df[~processed_df['participant_id'].isin(ids_failed_attention)]

# Save processed data
processed_df.to_csv('../data/processed/processed_item_level_data.csv', index=False)
processed_df.head(3)

Data shape after merging AI variables: (2288, 18)


,participant_id,condition,week,imageid,time_point,accuracy,belief,confidence,ground_truth,seen,ai_usage,ai_image_usage,ai_literacy
0,560379c9e372c00011bd4a90,Persuasive,0,0,before,1,0,100,fake,False,2-4 hours a day,Once a week,5.166667
1,560379c9e372c00011bd4a90,Persuasive,0,0,with,1,0,100,fake,False,2-4 hours a day,Once a week,5.166667
2,560379c9e372c00011bd4a90,Persuasive,0,100,after,1,0,100,fake,False,2-4 hours a day,Once a week,5.166667


In [53]:
# Summary statistics for AI variables
print("AI Usage distribution:")
print(processed_df['ai_usage'].value_counts())
print("\nAI Image Usage distribution:")
print(processed_df['ai_image_usage'].value_counts())
print(f"\nAI Literacy score statistics:")
print(processed_df['ai_literacy'].describe())
print(f"\nFinal processed dataset shape: {processed_df.shape}")
print(f"Columns in processed dataset: {list(processed_df.columns)}")

AI Usage distribution:
ai_usage
0-2 hours a day            1980
Once a week                1116
2-4 hours a day             684
Once a month                324
4-6 hours a day             216
More than 6 hours a day     144
Never                        72
Name: count, dtype: int64

AI Image Usage distribution:
ai_image_usage
Once a week     1152
Rarely          1080
Never           1044
Everyday         828
Once a month     432
Name: count, dtype: int64

AI Literacy score statistics:
count    4500.000000
mean        5.628267
std         0.874118
min         3.000000
25%         5.166667
50%         5.833333
75%         6.166667
max         7.000000
Name: ai_literacy, dtype: float64

Final processed dataset shape: (4536, 13)
Columns in processed dataset: ['participant_id', 'condition', 'week', 'imageid', 'time_point', 'accuracy', 'belief', 'confidence', 'ground_truth', 'seen', 'ai_usage', 'ai_image_usage', 'ai_literacy']
